# Feature Engineering Strategies — AI Lab Instructor Notebook

*Evaluation & Debugging · Medium*

Practice the feature engineering moves that win interviews and ship in production: missing data handling, categorical encoding, nonlinear transforms, and interaction features—*without leaking labels*.

**Outcome.** Students can:
- build leakage-safe feature pipelines with sklearn,
- choose appropriate encoding strategies,
- add meaningful interaction features,
- validate that features help via disciplined evaluation.

7 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Feature Engineering Strategies — Student Lab (Titanic)

Focus: missing data, encoding, nonlinear transforms, and interactions — **leakage-safe**.

## Section 0 — Load Titanic (Kaggle) with fallback
Expected: `data/titanic/train.csv`

If missing, a tiny synthetic dataset is used so the notebook runs.

In [ ]:
$10

**Why this works.** # Feature Engineering Strategies — Instructor (Titanic)

## Task 2: Train a baseline with leakage-safe preprocessing
*Section: Baseline pipeline*

## Section 1 — Baseline pipeline

### Task 1.1: Train a baseline with leakage-safe preprocessing

Use:
- Numeric: Age, Fare, SibSp, Parch
- Categorical: Pclass, Sex, Embarked

- Create X/y
- Build ColumnTransformer (impute + scale numeric, impute + one-hot categorical)
- Evaluate LogisticRegression with StratifiedKFold CV

**Checkpoint:** Why is the pipeline necessary to avoid leakage in CV?

In [ ]:
$11

## Task 3: Add missingness indicator features
*Section: Missing data: indicators*

## Section 2 — Missing data: indicators

### Task 2.1: Add missingness indicator features

Add: Age_is_missing, Cabin_is_missing

- Create these columns
- Add them to numeric pipeline
- Re-run CV and compare

**Checkpoint:** Why can missingness itself be predictive?

In [ ]:
X2 = X.copy()
X2['Age_is_missing'] = X2['Age'].isna().astype(int)
X2['Cabin_is_missing'] = X2['Cabin'].isna().astype(int)
numeric_features2 = numeric_features + ['Age_is_missing','Cabin_is_missing']
preprocess2 = make_preprocess(numeric_features2, categorical_features)
pipe2 = Pipeline(steps=[('preprocess', preprocess2), ('model', LogisticRegression(max_iter=5000))])
scores_ind = cv_acc(pipe2, X2, y)
print('with indicators acc mean', np.mean(scores_ind['test_acc']), 'std', np.std(scores_ind['test_acc']))

# Rationale: missingness can be informative (not missing-at-random).

**Why this works.** missingness can be informative (not missing-at-random).

## Task 4: Extract Title from Name and one-hot encode
*Section: Encoding: One-hot vs Target encoding*

## Section 3 — Encoding: One-hot vs Target encoding

### Task 3.1: Extract Title from Name and one-hot encode

Example titles: Mr, Mrs, Miss, Master.

- Extract Title with regex
- Add it as categorical feature
- Re-run CV

In [ ]:
def extract_title(name: str):
    if not isinstance(name, str):
        return 'Unknown'
    m = re.search(r',\s*([^\.]*)\.', name)
    return m.group(1).strip() if m else 'Unknown'

X3 = X2.copy()
X3['Title'] = X3['Name'].apply(extract_title)

categorical_features3 = categorical_features + ['Title']
numeric_features3 = numeric_features2
preprocess3 = make_preprocess(numeric_features3, categorical_features3)
pipe3 = Pipeline(steps=[('preprocess', preprocess3), ('model', LogisticRegression(max_iter=5000))])
scores3 = cv_acc(pipe3, X3, y)
print('with Title acc mean', np.mean(scores3['test_acc']))

# Rationale: Title captures social status correlated with survival.

**Why this works.** Title captures social status correlated with survival.

## Task 5: Naive target encoding (demonstrate leakage)
*Section: Encoding: One-hot vs Target encoding*

### Task 3.2: Naive target encoding (demonstrate leakage)

We will *intentionally* do something wrong: compute mean Survived per Title using the full dataset, then map it back.

- Create Title_target_mean using full data (this is leakage)
- Compare CV score and explain why it inflates

**Checkpoint:** How do you fix target encoding correctly?

In [ ]:
X4 = X3.copy()
title_mean = y.groupby(X3['Title']).mean()
X4['Title_target_mean_leaky'] = X4['Title'].map(title_mean).fillna(y.mean())

numeric_features4 = numeric_features3 + ['Title_target_mean_leaky']
categorical_features4 = categorical_features  # drop Title to keep it purely numeric encoding
preprocess4 = make_preprocess(numeric_features4, categorical_features4)
pipe4 = Pipeline(steps=[('preprocess', preprocess4), ('model', LogisticRegression(max_iter=5000))])
scores4 = cv_acc(pipe4, X4, y)
print('LEAKY target encoding acc mean', np.mean(scores4['test_acc']))

# Rationale: leakage occurs because encoding uses labels from validation folds. Fix with out-of-fold encoding.

**Why this works.** leakage occurs because encoding uses labels from validation folds. Fix with out-of-fold encoding.

## Task 6: Log-transform Fare
*Section: Nonlinear transforms + binning*

## Section 4 — Nonlinear transforms + binning

### Task 4.1: Log-transform Fare

In [ ]:
X5 = X3.copy()
X5['Fare_log'] = np.log1p(pd.to_numeric(X5['Fare'], errors='coerce'))

numeric_features5 = [c for c in numeric_features3 if c != 'Fare'] + ['Fare_log']
categorical_features5 = categorical_features3
preprocess5 = make_preprocess(numeric_features5, categorical_features5)
pipe5 = Pipeline(steps=[('preprocess', preprocess5), ('model', LogisticRegression(max_iter=5000))])
scores5 = cv_acc(pipe5, X5, y)
print('with Fare_log acc mean', np.mean(scores5['test_acc']))

# Rationale: Fare is heavy-tailed; log reduces scale sensitivity for linear models.

**Why this works.** Fare is heavy-tailed; log reduces scale sensitivity for linear models.

## Task 7: FamilySize and IsAlone
*Section: Interaction features*

## Section 5 — Interaction features

### Task 5.1: FamilySize and IsAlone
FamilySize = SibSp + Parch + 1
IsAlone = FamilySize == 1

### Task 5.2: Sex × Pclass cross
Create a categorical cross feature like `Sex_Pclass`.

In [ ]:
X6 = X5.copy()
X6['FamilySize'] = pd.to_numeric(X6['SibSp'], errors='coerce').fillna(0) + pd.to_numeric(X6['Parch'], errors='coerce').fillna(0) + 1
X6['IsAlone'] = (X6['FamilySize'] == 1).astype(int)
X6['Sex_Pclass'] = X6['Sex'].astype(str) + '_P' + X6['Pclass'].astype(str)

numeric_features6 = numeric_features5 + ['FamilySize','IsAlone']
categorical_features6 = categorical_features5 + ['Sex_Pclass']
preprocess6 = make_preprocess(numeric_features6, categorical_features6)
pipe6 = Pipeline(steps=[('preprocess', preprocess6), ('model', LogisticRegression(max_iter=5000))])
scores6 = cv_acc(pipe6, X6, y)
print('with interactions acc mean', np.mean(scores6['test_acc']))

# Rationale: interactions let linear models represent conditional effects (e.g., Sex effect varies by Pclass).

**Why this works.** interactions let linear models represent conditional effects (for example, Sex effect varies by Pclass).